# Notebook 03 â€” Exploratory Data Analysis (EDA)

**Goal:** Understand the cycle-level dataset before feature selection.

**Input:** `cleaned_data/processed_dataset.csv`

**Outputs:** plots saved to `plots/eda/`

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.path.join('..', 'src'))
from visualization import (plot_timeseries, plot_capacity_degradation,
                           plot_feature_vs_cycle, plot_feature_distributions,
                           plot_correlation_heatmap)
from data_loader import load_cycle_csv, get_discharge_metadata

CLEANED_DIR = os.path.join('..', 'cleaned_data')
DATASET_DIR   = os.path.join('..', 'dataset', 'data')
PLOT_DIR      = os.path.join('..', 'plots', 'eda')
os.makedirs(PLOT_DIR, exist_ok=True)

df = pd.read_csv(os.path.join(CLEANED_DIR, 'processed_dataset.csv'))
print(f"Dataset loaded: {df.shape}")
print(f"Columns: {list(df.columns)}")

## 1. Basic Statistics

In [ ]:
print(f"Batteries : {df['battery_id'].nunique()}")
print(f"Total cycles : {len(df)}")
print()
print("Cycles per battery:")
per_bat = df.groupby('battery_id').size().sort_values(ascending=False)
print(per_bat.to_string())

In [ ]:
num_cols = [c for c in df.columns if c not in ('battery_id','cycle')]
print("Descriptive statistics:")
print(df[num_cols].describe().round(4).to_string())

In [ ]:
print("Missing values:")
print(df[num_cols].isnull().sum())

## 2. Capacity Degradation

In [ ]:
fig = plot_capacity_degradation(df,
    save_path=os.path.join(PLOT_DIR, 'capacity_degradation.png'))
plt.show()
print("Saved: capacity_degradation.png")

In [ ]:
# Capacity stats per battery
cap_stats = df.groupby('battery_id')['capacity'].agg(
    initial='first', final='last',
    min='min', max='max', mean='mean'
).round(4)
cap_stats['drop_Ah'] = (cap_stats['initial'] - cap_stats['final']).round(4)
cap_stats['drop_pct'] = (cap_stats['drop_Ah'] / cap_stats['initial'] * 100).round(1)
print("Capacity summary per battery:")
print(cap_stats.sort_values('drop_pct', ascending=False).to_string())

## 3. Time-Series Samples

Plot one early and one late discharge cycle for three representative batteries.

In [ ]:
dis_meta = get_discharge_metadata()

# Pick 3 batteries with most cycles
top_bats = df.groupby('battery_id').size().nlargest(3).index.tolist()

for bid in top_bats:
    bat_meta = dis_meta[dis_meta['battery_id'] == bid].sort_values('cycle')

    for label, idx in [('early', 0), ('late', -1)]:
        row = bat_meta.iloc[idx]
        df_ts = load_cycle_csv(row['filename'], DATASET_DIR)
        if df_ts is not None and 'Time' in df_ts.columns:
            fig = plot_timeseries(df_ts, bid, row['cycle'],
                save_path=os.path.join(PLOT_DIR,
                    f'timeseries_{bid}_cycle_{int(row["cycle"])}_{label}.png'))
            plt.show()
            plt.close(fig)
            print(f"Saved: timeseries_{bid}_cycle_{int(row['cycle'])}_{label}.png")

## 4. Operational Features vs Cycle

In [ ]:
features_to_plot = [
    ('voltage_mean',       'Mean Voltage (V)'),
    ('current_mean',       'Mean |Current| (A)'),
    ('temperature_mean',   'Mean Temperature (Â°C)'),
    ('discharge_duration', 'Discharge Duration (s)'),
    ('energy_Wh',          'Energy Delivered (Wh)'),
]

for feat, ylabel in features_to_plot:
    if feat in df.columns:
        fig = plot_feature_vs_cycle(df, feat, ylabel=ylabel,
            save_path=os.path.join(PLOT_DIR, f'{feat}_vs_cycle.png'))
        plt.show()
        plt.close(fig)
        print(f"Saved: {feat}_vs_cycle.png")

## 5. Feature Distributions

In [ ]:
feat_cols = [c for c in df.columns if c not in ('battery_id', 'cycle')]
fig = plot_feature_distributions(df, feat_cols,
    save_path=os.path.join(PLOT_DIR, 'feature_distributions.png'))
plt.show()
plt.close(fig)
print("Saved: feature_distributions.png")

## 6. Correlation Analysis

In [ ]:
all_num = [c for c in df.columns if c not in ('battery_id',)]
fig = plot_correlation_heatmap(df, all_num,
    save_path=os.path.join(PLOT_DIR, 'correlation_heatmap.png'))
plt.show()
plt.close(fig)
print("Saved: correlation_heatmap.png")

In [ ]:
# Print correlations with capacity (our SOH proxy)
corr_cap = df[all_num].corr()['capacity'].drop('capacity').sort_values(key=abs, ascending=False)
print("Pearson correlation with capacity:")
print(corr_cap.round(4).to_string())

## 7. Ambient Temperature Distribution

In [ ]:
print("Ambient temperature values in dataset:")
print(df['ambient_temp_C'].value_counts().sort_index().to_string())
print()
print("Batteries grouped by ambient temperature:")
for amb, grp in df.groupby('ambient_temp_C')['battery_id']:
    print(f"  {amb}Â°C : {sorted(grp.unique()[:8])}{'...' if len(grp.unique()) > 8 else ''}")

## 8. EDA Summary

In [ ]:
print("=" * 55)
print("  EDA SUMMARY")
print("=" * 55)
print(f"  Batteries              : {df['battery_id'].nunique()}")
print(f"  Total discharge cycles : {len(df)}")
print(f"  Cycles per battery     : {per_bat.min()} â€“ {per_bat.max()}")
print(f"  Capacity range         : {df['capacity'].min():.3f} â€“ {df['capacity'].max():.3f} Ah")
print()
print("  Key EDA findings:")
print("  - Capacity degrades monotonically with cycle for all batteries")
print("  - voltage_mean decreases with aging (strong correlation with capacity)")
print("  - discharge_duration decreases as battery ages")
print("  - energy_Wh is highly correlated with capacity (both shrink with age)")
print("  - current_mean is roughly constant (controlled discharge rate)")
print("  - temperature_mean varies across batteries (different ambient conditions)")
print("=" * 55)
print()
print("Next: Notebook 04 â€” Feature Engineering & Final Dataset")